# CSC-350 Artificial Intelligence Semester Project (Section E)
## Advanced Time-Series Forecasting using Lorentzian Distance Classification & Kernel Regression

**Instructor:** Engr. Muhammad Irfan Younas Mughal (eMIYM)  
**Group Members:**
1. **Tayyab Mangi** (CMS: 023-24-0118) - Coded model training, custom Lorentzian Scikit-Learn metric, backtesting, results report, and slides.
2. **Asif Ali Rattar** (CMS: 023-24-0158) - Coded Binance Klines API data fetcher, indicator engineering, plotting scripts, introduction, and literature review.

---

In [ ]:
# === Google Colab Setup / Library check ===
import os
import sys
import requests
import time
import pickle
import zipfile
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

### Part 1: Binance Candlestick (Klines) Data Fetching
We fetch 5,000 historical 5-minute candles of the BTC/USDT spot market from the Binance API. The data is paginated backwards in time using the `endTime` parameter.

In [ ]:
import os
import requests
import pandas as pd
import time

def fetch_binance_klines(symbol="BTCUSDT", interval="5m", limit=1000, end_time=None):
    """
    Fetch a single batch of klines from the Binance API.
    Uses endTime to allow backward pagination.
    """
    url = "https://api.binance.com/api/v3/klines"
    params = {
        "symbol": symbol,
        "interval": interval,
        "limit": limit
    }
    if end_time:
        params["endTime"] = end_time
    
    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json()

def fetch_historical_dataset(symbol="BTCUSDT", interval="5m", total_candles=5000):
    """
    Fetch a large historical dataset of klines by looping in batches of 1000 backwards in time.
    """
    print(f"Fetching {total_candles} candles for {symbol} ({interval}) from Binance...")
    all_klines = []
    current_end = None
    
    batch_size = 1000
    while len(all_klines) < total_candles:
        remaining = total_candles - len(all_klines)
        limit = min(batch_size, remaining)
        
        try:
            klines = fetch_binance_klines(symbol, interval, limit, current_end)
            if not klines:
                break
            
            # Prepend the older klines to keep chronological order
            all_klines = klines + all_klines
            
            # The next batch must end 1ms before the oldest candle in the current batch
            first_open_time = klines[0][0]
            current_end = first_open_time - 1
            
            print(f"Fetched {len(all_klines)}/{total_candles} candles...")
            # Small delay to respect API limits
            time.sleep(0.1)
        except Exception as e:
            print(f"Error fetching data from Binance: {e}")
            break
            
    # Format raw API response into a pandas DataFrame
    columns = [
        "open_time", "open", "high", "low", "close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
    ]
    # In case we fetched slightly more due to batch alignment, slice to match requested amount
    df = pd.DataFrame(all_klines[-total_candles:], columns=columns)
    
    # Cast numerical fields to float
    numeric_cols = ["open", "high", "low", "close", "volume"]
    for col in numeric_cols:
        df[col] = df[col].astype(float)
        
    # Convert timestamps to human-readable dates
    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms")
    df["close_time"] = pd.to_datetime(df["close_time"], unit="ms")
    
    return df

### Part 2: Feature Engineering & Technical Indicators
We calculate standard indicators (RSI, WaveTrend, CCI, ADX) and a **causal, non-repainting Nadaraya-Watson Rational Quadratic Kernel Regression** filter to smooth out macro price movements.

In [ ]:
import numpy as np
import pandas as pd

def ema(series, period):
    """
    Calculate the Exponential Moving Average (EMA).
    """
    return series.ewm(span=period, adjust=False).mean()

def sma(series, period):
    """
    Calculate the Simple Moving Average (SMA).
    """
    return series.rolling(window=period).mean()

def rsi(series, period=14):
    """
    Calculate the Relative Strength Index (RSI) using Wilder's smoothing.
    """
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    
    # Wilder's smoothing uses alpha = 1 / period, equivalent to com = period - 1
    avg_gain = gain.ewm(com=period-1, adjust=False).mean()
    avg_loss = loss.ewm(com=period-1, adjust=False).mean()
    
    rs = avg_gain / (avg_loss + 1e-10)
    return 100 - (100 / (1 + rs))

def wavetrend(high, low, close, n1=10, n2=11):
    """
    Calculate the WaveTrend Oscillator (by LazyBear).
    Returns (WT1, WT2) where WT1 is the main line and WT2 is the signal line.
    """
    ap = (high + low + close) / 3
    esa = ema(ap, n1)
    d = ema((ap - esa).abs(), n1)
    ci = (ap - esa) / (0.015 * d + 1e-10)
    wt1 = ema(ci, n2)
    wt2 = sma(wt1, 4)
    return wt1, wt2

def cci(high, low, close, period=20):
    """
    Calculate the Commodity Channel Index (CCI).
    """
    tp = (high + low + close) / 3
    tp_sma = sma(tp, period)
    # Mean Absolute Deviation (MAD)
    mad = tp.rolling(window=period).apply(
        lambda x: np.mean(np.abs(x - np.mean(x))), raw=True
    )
    return (tp - tp_sma) / (0.015 * mad + 1e-10)

def adx(high, low, close, period=14):
    """
    Calculate the Average Directional Index (ADX) using Wilder's smoothing.
    """
    # True Range (TR)
    tr1 = high - low
    tr2 = (high - close.shift(1)).abs()
    tr3 = (low - close.shift(1)).abs()
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    
    # Directional Movement (+DM and -DM)
    up_move = high.diff()
    down_move = low.shift(1) - low
    
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
    
    plus_dm = pd.Series(plus_dm, index=close.index)
    minus_dm = pd.Series(minus_dm, index=close.index)
    
    # Smooth TR and DMs
    atr = tr.ewm(com=period-1, adjust=False).mean()
    plus_di = 100 * plus_dm.ewm(com=period-1, adjust=False).mean() / (atr + 1e-10)
    minus_di = 100 * minus_dm.ewm(com=period-1, adjust=False).mean() / (atr + 1e-10)
    
    # Calculate DX and smooth it to get ADX
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di + 1e-10)
    adx_series = dx.ewm(com=period-1, adjust=False).mean()
    return adx_series

def nadaraya_watson_rational_quadratic(close, h=8, r=8.0, lookback=25):
    """
    Calculate a causal (non-repainting) Nadaraya-Watson Kernel Regression
    using the Rational Quadratic Kernel.
    
    h: lookback window (bandwidth)
    r: relative weighting parameter (alpha)
    lookback: sliding window size for local regression
    """
    # Compute kernel weights for historical offsets (0 to lookback-1)
    weights = np.zeros(lookback)
    for i in range(lookback):
        weights[i] = (1 + (i**2) / (2 * r * (h**2)))**(-r)
        
    weights_sum = np.sum(weights)
    
    result = np.zeros(len(close))
    close_arr = close.values
    
    # Compute the weighted average for each bar in a sliding window
    for t in range(len(close)):
        if t < lookback - 1:
            result[t] = close_arr[t]  # fallback for early bars
        else:
            # Take the window of closing prices leading up to t, reversed
            window = close_arr[t - lookback + 1 : t + 1][::-1]
            result[t] = np.sum(window * weights) / weights_sum
            
    return pd.Series(result, index=close.index)


### Part 3: Custom Lorentzian Classifier & Feature Preparation
Here we implement the custom **Lorentzian Distance formula** and build the training/testing classification vectors with binary targets (1: price Up, -1: price Down/Flat).

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler


# Ensure local imports work correctly



def lorentzian_distance(x, y):
    """
    Custom Lorentzian distance metric.
    Calculates logarithmic warping of coordinates to dampen outlier impacts.
    """
    return np.sum(np.log(1 + np.abs(x - y)))

def prepare_features_and_targets(df, target_horizon=1, threshold=0.0002):
    """
    Calculate technical indicators and create feature vectors/targets.
    Target labels predict price direction over the next target_horizon bars:
    -  1 (Up): close price increases by more than the threshold percentage
    - -1 (Down): close price decreases by more than the threshold percentage
    -  0 (Neutral): close price remains flat / within threshold
    """
    df = df.copy()
    
    # 1. Feature Engineering
    df["f1_rsi"] = rsi(df["close"], period=14)
    wt1, wt2 = wavetrend(df["high"], df["low"], df["close"], n1=10, n2=11)
    df["f2_wt"] = wt1
    df["f3_cci"] = cci(df["high"], df["low"], df["close"], period=20)
    df["f4_adx"] = adx(df["high"], df["low"], df["close"], period=14)
    df["f5_rsi_short"] = rsi(df["close"], period=9)
    
    # Custom non-parametric kernel filter for trend confirmation
    df["kernel_reg"] = nadaraya_watson_rational_quadratic(df["close"], h=8, r=8.0, lookback=25)
    df["kernel_slope"] = df["kernel_reg"].diff()  # Positive is bullish, negative is bearish
    
    # 2. Target Labeling (Next 5-minute price direction)
    # The return over the next target_horizon bars
    future_change = (df["close"].shift(-target_horizon) - df["close"]) / df["close"]
    
    # For a clean binary classification problem, we assign 1 to positive changes and -1 to negative or zero changes
    targets = np.where(future_change > threshold, 1, -1)
    df["target"] = targets
    
    # Drop rows containing NaNs arising from lagging window operations and the future shifted targets
    df_clean = df.dropna().copy()
    
    # Align features (X) and labels (y)
    feature_cols = ["f1_rsi", "f2_wt", "f3_cci", "f4_adx", "f5_rsi_short"]
    X = df_clean[feature_cols].values
    y = df_clean["target"].values
    
    return df_clean, X, y

def train_and_evaluate_models(X_train, X_test, y_train, y_test):
    """
    Train and compare the custom Lorentzian KNN against standard baseline classifiers.
    """
    # Scaling is crucial for distance-based ML models. RobustScaler is used because it
    # relies on IQR to resist technical indicator outliers.
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 1. Baseline: Logistic Regression
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_train_scaled, y_train)
    
    # 2. Baseline: Random Forest
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train_scaled, y_train)
    
    # 3. Baseline: standard Euclidean KNN
    knn_euclidean = KNeighborsClassifier(n_neighbors=8, metric="euclidean")
    knn_euclidean.fit(X_train_scaled, y_train)
    
    # 4. Custom: Physics-Inspired Lorentzian KNN using a custom callable distance metric
    knn_lorentzian = KNeighborsClassifier(n_neighbors=8, metric=lorentzian_distance)
    knn_lorentzian.fit(X_train_scaled, y_train)
    
    models = {
        "Logistic Regression": (lr, X_test_scaled),
        "Random Forest": (rf, X_test_scaled),
        "Euclidean KNN": (knn_euclidean, X_test_scaled),
        "Lorentzian KNN": (knn_lorentzian, X_test_scaled)
    }
    
    results = {}
    for name, (model, X_t) in models.items():
        preds = model.predict(X_t)
        acc = np.mean(preds == y_test)
        results[name] = {
            "model": model,
            "predictions": preds,
            "accuracy": acc
        }
        
    return results, scaler

### Part 4: Classification Pipeline, Model Training, & Backtesting
We execute model training comparing Lorentzian KNN to Euclidean KNN, Random Forest, and Logistic Regression. We run a trading simulation (strategy vs. Buy & Hold BTC) adjusted for 0.05% transaction fees.

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import RobustScaler


# Ensure local imports work correctly



def run_pipeline():
    """
    Main pipeline to load the Binance dataset, build features, train the classifiers,
    generate performance evaluation metrics/plots, and backtest the trading strategy.
    """
    print("Starting ML time-series forecasting pipeline...")
    os.makedirs("results", exist_ok=True)
    os.makedirs("model", exist_ok=True)
    
    # 1. Load the historical Binance dataset
    csv_path = os.path.join("dataset", "btc_5m_historical.csv")
    if not os.path.exists(csv_path):
        print(f"Error: Dataset {csv_path} not found. Please run binance_klines_fetcher.py first.")
        return
    
    df = pd.read_csv(csv_path)
    
    # 2. Preprocess data
    # threshold=0.0 sets up a clean 2-class problem (Up vs Down/Flat) for simplified evaluation
    print("Engineering features and target labels...")
    df_clean, X, y = prepare_features_and_targets(df, target_horizon=1, threshold=0.0)
    
    # Output class distributions
    unique, counts = np.unique(y, return_counts=True)
    class_balance = dict(zip(unique, counts))
    print(f"Class balance: {class_balance} (-1.0: Down/Flat, 1.0: Up)")
    
    # 3. Train-Test Split (80-20 chronological split to avoid data leakage)
    split_idx = int(len(X) * 0.8)
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]
    
    test_df = df_clean.iloc[split_idx:].copy()
    
    # 4. Train and evaluate models
    print("Training models and computing performance...")
    results, scaler = train_and_evaluate_models(X_train, X_test, y_train, y_test)
    
    # Save the scaler and the best model (Lorentzian KNN) for modular packaging
    best_model_name = "Lorentzian KNN"
    best_model = results[best_model_name]["model"]
    
    with open(os.path.join("model", "scaler.pkl"), "wb") as f:
        pickle.dump(scaler, f)
    with open(os.path.join("model", "lorentzian_knn_model.pkl"), "wb") as f:
        pickle.dump(best_model, f)
    print(f"Successfully saved scaler and best model ({best_model_name}) to model/ directory!")
    
    # 5. Output Accuracies and classification report
    print("\n" + "="*30)
    print("       MODEL ACCURACIES")
    print("="*30)
    for name, res in results.items():
        print(f"{name:<20} Accuracy: {res['accuracy']:.4f}")
    print("="*30)
        
    print("\nClassification Report for Lorentzian KNN:")
    lorentzian_preds = results["Lorentzian KNN"]["predictions"]
    print(classification_report(y_test, lorentzian_preds, target_names=["Down/Flat", "Up"]))
    
    # Generate Confusion Matrix
    print("Generating Confusion Matrix plot...")
    cm = confusion_matrix(y_test, lorentzian_preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="GnBu", 
                xticklabels=["Down/Flat", "Up"], yticklabels=["Down/Flat", "Up"], 
                annot_kws={"size": 14})
    plt.title("Confusion Matrix: Lorentzian KNN Classifier", fontsize=13, fontweight="bold", pad=15)
    plt.xlabel("Predicted Label", fontsize=11, labelpad=10)
    plt.ylabel("True Label", fontsize=11, labelpad=10)
    plt.tight_layout()
    plt.savefig(os.path.join("results", "confusion_matrix.png"), dpi=300)
    plt.close()
    
    # 6. Generate ROC-AUC Curve Comparisons
    print("Generating ROC-AUC Curve plot...")
    plt.figure(figsize=(8, 6))
    
    X_test_scaled = scaler.transform(X_test)
    
    for name, res in results.items():
        model = res["model"]
        if hasattr(model, "predict_proba"):
            # Probability for class index 1 (representing label 1.0 / 'Up')
            probs = model.predict_proba(X_test_scaled)[:, 1]
            fpr, tpr, _ = roc_curve(y_test, probs, pos_label=1.0)
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.4f})", linewidth=2)
            
    plt.plot([0, 1], [0, 1], "k--", label="Random Guess (AUC = 0.5000)", linewidth=1.5)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate", fontsize=11, labelpad=10)
    plt.ylabel("True Positive Rate", fontsize=11, labelpad=10)
    plt.title("ROC-AUC Curves: Model Performance Comparison", fontsize=13, fontweight="bold", pad=15)
    plt.legend(loc="lower right", fontsize=10)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join("results", "roc_auc_curve.png"), dpi=300)
    plt.close()
    
    # 7. Simulated trading backtest
    print("Running simulated trading backtest on test data...")
    
    # Calculate simple close-to-close returns on the test dataframe
    test_df["returns"] = test_df["close"].pct_change()
    
    # Set our position signals: 1 for Long, -1 for Short
    test_df["signal"] = lorentzian_preds
    
    # The return in period t+1 is the signal in period t multiplied by close return in period t+1
    test_df["strategy_returns"] = test_df["signal"].shift(1) * test_df["returns"]
    
    # Standard transaction fee for Binance trades (0.05% per trade)
    # We pay a fee whenever we change our position (signal shifts)
    position_changes = test_df["signal"].diff().abs()
    transaction_fee = 0.0005  # 0.05%
    test_df["strategy_returns_net"] = test_df["strategy_returns"] - (position_changes * transaction_fee).fillna(0)
    
    # Calculate cumulative percentage returns
    test_df["cum_returns_bh"] = (1 + test_df["returns"].fillna(0)).cumprod() - 1
    test_df["cum_returns_strategy_gross"] = (1 + test_df["strategy_returns"].fillna(0)).cumprod() - 1
    test_df["cum_returns_strategy_net"] = (1 + test_df["strategy_returns_net"].fillna(0)).cumprod() - 1
    
    # Plotting Equity Curves
    plt.figure(figsize=(10, 6))
    time_series = test_df["close_time"]
    
    plt.plot(time_series, test_df["cum_returns_strategy_net"] * 100, 
             label="Lorentzian KNN Strategy (Net of Fees)", color="#009988", linewidth=2.2)
    plt.plot(time_series, test_df["cum_returns_strategy_gross"] * 100, 
             label="Lorentzian KNN Strategy (Gross)", color="#009988", linestyle="--", alpha=0.6, linewidth=1.2)
    plt.plot(time_series, test_df["cum_returns_bh"] * 100, 
             label="Buy & Hold BTC Benchmark", color="#CC3311", linewidth=1.8)
    
    plt.title("Backtest Equity Curves: Lorentzian KNN Strategy vs. BTC Benchmark", fontsize=13, fontweight="bold", pad=15)
    plt.xlabel("Date / Time", fontsize=11, labelpad=10)
    plt.ylabel("Cumulative Returns (%)", fontsize=11, labelpad=10)
    plt.legend(loc="upper left", fontsize=10)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    
    # Save the plot
    plt.savefig(os.path.join("results", "equity_curve.png"), dpi=300)
    plt.close()
    
    # Save the detailed test table
    test_df.to_csv(os.path.join("results", "backtest_results.csv"), index=False)
    print("Pipeline successfully completed! All results and premium plots saved in 'results/' and 'model/' directories.")

### Part 5: Project Submission ZIP Packager
We define the zipping manager to bundle all required directories into the standard required submission format.

In [ ]:
import os
import zipfile

def package_project():
    zip_filename = "AI_Project_GroupE_LorentzianClassification.zip"
    print(f"Starting packaging of semester project into: {zip_filename}...")
    
    # List of directories to include (exactly per submission guidelines)
    dirs_to_include = [
        "report",
        "slides",
        "code",
        "results",
        "dataset",
        "model",
        "contribution"
    ]

    
    # Ensure all directories exist
    for d in dirs_to_include:
        if not os.path.exists(d):
            print(f"Warning: Directory {d} does not exist. Creating it.")
            os.makedirs(d, exist_ok=True)
            
    # Create the zip archive
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for folder in dirs_to_include:
            print(f"Adding folder: {folder}/")
            for root, dirs, files in os.walk(folder):
                # Exclude __pycache__ folders
                if "__pycache__" in root:
                    continue
                for file in files:
                    file_path = os.path.join(root, file)
                    # Keep relative path starting with the directory name itself
                    arcname = os.path.relpath(file_path, start=os.path.dirname(folder))
                    zipf.write(file_path, arcname)
                    print(f"  + {arcname}")
                    
    print(f"\nSuccessfully created and packaged zip archive: {zip_filename}")
    size_mb = os.path.getsize(zip_filename) / (1024 * 1024)
    print(f"Zip file size: {size_mb:.2f} MB")

### Part 6: Master Pipeline Execution
Let's execute all components in sequence! We fetch raw candlestick data, preprocess, train the classifiers, generate high-resolution evaluation charts, and bundle everything into a ZIP file ready for E-Learning upload.

In [ ]:
# Create output directories
os.makedirs("dataset", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("model", exist_ok=True)
os.makedirs("report", exist_ok=True)
os.makedirs("slides", exist_ok=True)
os.makedirs("plagiarism", exist_ok=True)
os.makedirs("contribution", exist_ok=True)

# 1. Fetch 5000 candles from Binance
df_klines = fetch_historical_dataset(symbol="BTCUSDT", interval="5m", total_candles=5000)
df_klines.to_csv("dataset/btc_5m_historical.csv", index=False)
print("✓ Step 1: Binance dataset fetched and saved!\n")

# 2. Run training, evaluation & trading backtest
run_pipeline()
print("✓ Step 2: Training & backtesting pipeline completed successfully!\n")

# 3. Package into standard ZIP archive
package_project()
print("✓ Step 3: Submission package prepared and ready for upload!")

### Part 8: Display Generated Plot Graphics Inline
We load and render the high-resolution charts generated during the model evaluations and strategy backtests.

In [ ]:
from IPython.display import Image, display

print("--- Lorentzian KNN Confusion Matrix ---")
display(Image(filename="results/confusion_matrix.png"))

print("\n--- Multi-Model ROC-AUC Comparison ---")
display(Image(filename="results/roc_auc_curve.png"))

print("\n--- Cumulative Strategy Returns vs. Benchmark BTC Buy & Hold ---")
display(Image(filename="results/equity_curve.png"))